# 03 — Train and evaluate a small TCN

Upload one or both landmark ZIPs produced by notebooks 01 and 02. This notebook:

1. audits the manifests and landmark shapes,
2. makes fixed temporal windows,
3. splits by `group_id` before augmentation,
4. trains a small Temporal Convolutional Network,
5. reports accuracy, balanced accuracy, Macro F1, per-class metrics, and a confusion matrix,
6. downloads the best model and its complete report.

The Workout/Fitness dataset has no reliable person IDs. Its clip groups stop windows from the same video leaking across splits, but its score is still a development result. MM-Fit workout IDs provide the stronger person-separated evidence.


In [ ]:
%pip install -q --upgrade scikit-learn seaborn

In [ ]:
from pathlib import Path
import json
import os
import random
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from google.colab import files
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
WINDOW = 64
STRIDE = 32
BATCH_SIZE = 64
MAX_EPOCHS = 40
PATIENCE = 8

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
from google.colab import files

# Choose both ZIPs in the upload dialog. They are usually only a few MB.
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
assert zip_names, "Upload at least one landmark ZIP from notebook 01 or 02."

SOURCE_ROOT = Path("/content/landmark_sources")
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir()

for index, name in enumerate(zip_names):
    destination = SOURCE_ROOT / f"source_{index}"
    destination.mkdir()
    with zipfile.ZipFile(name) as archive:
        archive.extractall(destination)
    print(name, "->", destination)


In [ ]:
# Load every manifest and resolve each NPZ relative to that manifest.
frames = []
for manifest_path in SOURCE_ROOT.rglob("manifest.csv"):
    frame = pd.read_csv(manifest_path).fillna("")
    frame["npz_path"] = frame["output_path"].map(lambda value: str(manifest_path.parent / value))
    frames.append(frame)

assert frames, "No manifest.csv was found inside the uploaded ZIP(s)."
manifest = pd.concat(frames, ignore_index=True)
manifest = manifest[(manifest["status"] == "ok") & manifest["npz_path"].map(lambda p: Path(p).exists())].copy()

if "group_id" not in manifest or manifest["group_id"].eq("").any():
    manifest["group_id"] = manifest.apply(
        lambda row: f"{row['source_dataset']}:{row['clip_id']}", axis=1
    )

print("Valid clips:", len(manifest))
print("Datasets:")
print(manifest["source_dataset"].value_counts())
print()
print("Classes:")
print(manifest["canonical_class"].value_counts().sort_index())
print()
print("Unique groups:", manifest["group_id"].nunique())

## Windows

The TCN needs equal-length batches. A 64-frame window represents about 4.3 seconds at 15 FPS. Long clips create overlapping windows; short clips are padded by repeating their final frame. Every window keeps its original clip/person group, so related windows stay in one split.


In [ ]:
def resample_sequence(sequence, source_fps, target_fps=15):
    if not source_fps or abs(float(source_fps) - target_fps) < 0.1:
        return sequence
    duration = (len(sequence) - 1) / float(source_fps)
    target_length = max(2, int(round(duration * target_fps)) + 1)
    old_x = np.linspace(0, duration, len(sequence))
    new_x = np.linspace(0, duration, target_length)
    flat = sequence.reshape(len(sequence), -1)
    sampled = np.stack([np.interp(new_x, old_x, flat[:, i]) for i in range(flat.shape[1])], axis=1)
    return sampled.reshape(target_length, 17, 4).astype(np.float32)


windows, labels, groups, clip_ids, sources = [], [], [], [], []
for row in manifest.itertuples(index=False):
    with np.load(row.npz_path, allow_pickle=False) as item:
        sequence = item["landmarks"].astype(np.float32)
    if sequence.ndim != 3 or sequence.shape[1:] != (17, 4):
        print("Skipping unexpected shape:", row.npz_path, sequence.shape)
        continue
    sequence = resample_sequence(sequence, getattr(row, "source_fps", 15) or 15)
    starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
    if len(sequence) > WINDOW and starts[-1] != len(sequence) - WINDOW:
        starts.append(len(sequence) - WINDOW)
    for start in starts:
        window = sequence[start:start + WINDOW]
        if len(window) < WINDOW:
            window = np.concatenate([window, np.repeat(window[-1:], WINDOW - len(window), axis=0)])
        windows.append(window)
        labels.append(row.canonical_class)
        groups.append(row.group_id)
        clip_ids.append(row.clip_id)
        sources.append(row.source_dataset)

X = np.asarray(windows, dtype=np.float32)
labels = np.asarray(labels)
groups = np.asarray(groups)
clip_ids = np.asarray(clip_ids)
sources = np.asarray(sources)
CLASSES = sorted(np.unique(labels).tolist())
class_to_id = {name: index for index, name in enumerate(CLASSES)}
y = np.asarray([class_to_id[name] for name in labels], dtype=np.int64)

print("Windows:", X.shape)
print("Classes:", CLASSES)
print(pd.Series(labels).value_counts().sort_index())

In [ ]:
# Fold 0 is test. Fold 0 of the remaining data is validation.
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_val_idx, test_idx = next(outer.split(X, y, groups))

inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED + 1)
inner_train, inner_val = next(inner.split(X[train_val_idx], y[train_val_idx], groups[train_val_idx]))
train_idx = train_val_idx[inner_train]
val_idx = train_val_idx[inner_val]

for name, idx in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    print(name, "windows=", len(idx), "groups=", len(np.unique(groups[idx])), "classes=", len(np.unique(y[idx])))

assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert not (set(groups[val_idx]) & set(groups[test_idx]))

In [ ]:
class PoseWindowDataset(Dataset):
    def __init__(self, indices, training=False):
        self.indices = np.asarray(indices)
        self.training = training

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual = self.indices[index]
        sample = X[actual].copy()
        if self.training:
            # Light pose noise improves robustness without changing the action label.
            sample[:, :, :3] += np.random.normal(0, 0.01, sample[:, :, :3].shape).astype(np.float32)
        # Use shared x/y only. MM-Fit has no MediaPipe z/visibility values, so
        # including them would let the network identify the dataset source.
        sample = sample[:, :, :2].reshape(WINDOW, -1).T
        return torch.from_numpy(sample), torch.tensor(y[actual], dtype=torch.long)


train_counts = np.bincount(y[train_idx], minlength=len(CLASSES))
sample_weights = 1.0 / np.maximum(train_counts[y[train_idx]], 1)
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(PoseWindowDataset(train_idx, True), batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(PoseWindowDataset(val_idx), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(PoseWindowDataset(test_idx), batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks = []
        current = input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        temporal_features = self.features(x)
        return self.classifier(temporal_features.mean(dim=-1))


model = SmallTCN(17 * 2, len(CLASSES)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def run_epoch(loader, training=False):
    model.train(training)
    losses, truths, predictions = [], [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for features, targets in loader:
            features, targets = features.to(DEVICE), targets.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = criterion(logits, targets)
            if training:
                loss.backward()
                optimizer.step()
            losses.append(loss.item() * len(targets))
            truths.extend(targets.cpu().numpy())
            predictions.extend(logits.argmax(1).cpu().numpy())
    return sum(losses) / len(truths), f1_score(truths, predictions, average="macro", zero_division=0)


history = []
best_f1 = -1.0
epochs_without_improvement = 0
BEST_PATH = Path("/content/best_tcn.pt")

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_f1 = run_epoch(train_loader, training=True)
    val_loss, val_f1 = run_epoch(val_loader, training=False)
    scheduler.step(val_f1)
    history.append({"epoch": epoch, "train_loss": train_loss, "train_macro_f1": train_f1, "val_loss": val_loss, "val_macro_f1": val_f1})
    print(f"{epoch:02d} train loss={train_loss:.4f} F1={train_f1:.3f} | val loss={val_loss:.4f} F1={val_f1:.3f}")
    if val_f1 > best_f1 + 1e-4:
        best_f1 = val_f1
        epochs_without_improvement = 0
        torch.save(model.state_dict(), BEST_PATH)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping")
            break

model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))

In [ ]:
def predict(loader):
    model.eval()
    truths, predictions, probabilities = [], [], []
    with torch.no_grad():
        for features, targets in loader:
            logits = model(features.to(DEVICE))
            probs = logits.softmax(1).cpu().numpy()
            truths.extend(targets.numpy())
            predictions.extend(probs.argmax(1))
            probabilities.extend(probs)
    return np.asarray(truths), np.asarray(predictions), np.asarray(probabilities)


y_true, y_pred, y_prob = predict(test_loader)
metrics = {
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    "seed": SEED,
    "window_frames": WINDOW,
    "stride_frames": STRIDE,
    "classes": CLASSES,
}
print(json.dumps(metrics, indent=2))
report = classification_report(
    y_true, y_pred, labels=np.arange(len(CLASSES)),
    target_names=CLASSES, output_dict=True, zero_division=0,
)
display(pd.DataFrame(report).T.round(3))

In [ ]:
REPORT_ROOT = Path("/content/calorie_tcn_report")
REPORT_ROOT.mkdir(exist_ok=True)

matrix = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASSES)))
plt.figure(figsize=(max(8, len(CLASSES) * 0.8), max(6, len(CLASSES) * 0.7)))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("TCN test confusion matrix")
plt.tight_layout()
plt.savefig(REPORT_ROOT / "confusion_matrix.png", dpi=180)
plt.show()

history_frame = pd.DataFrame(history)
history_frame.to_csv(REPORT_ROOT / "training_history.csv", index=False)
history_frame.plot(x="epoch", y=["train_macro_f1", "val_macro_f1"], figsize=(8, 4), grid=True)
plt.tight_layout()
plt.savefig(REPORT_ROOT / "learning_curve.png", dpi=180)
plt.show()

torch.save({
    "state_dict": model.state_dict(),
    "classes": CLASSES,
    "window_frames": WINDOW,
    "input_features": 34,
}, REPORT_ROOT / "tcn_model.pt")
(REPORT_ROOT / "metrics.json").write_text(json.dumps(metrics, indent=2))
(REPORT_ROOT / "classification_report.json").write_text(json.dumps(report, indent=2))

prediction_rows = manifest.iloc[0:0].copy()  # schema reminder; window predictions live below
pd.DataFrame({
    "clip_id": clip_ids[test_idx],
    "source_dataset": sources[test_idx],
    "true_class": [CLASSES[i] for i in y_true],
    "predicted_class": [CLASSES[i] for i in y_pred],
    "confidence": y_prob.max(axis=1),
}).to_csv(REPORT_ROOT / "test_window_predictions.csv", index=False)

zip_path = shutil.make_archive("/content/calorie_tcn_report", "zip", root_dir=REPORT_ROOT)
print("Report ZIP:", zip_path)
files.download(zip_path)